
  %load_ext autoreload : magic command de Jupyter de autoreload
  
  %autoreload 2 : recarga todo automáticamente(2)

In [112]:
%load_ext autoreload 
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [113]:
# PAQUETE LIBRERIAS ANALISIS DE DATOS y de SKLEARN
import pandas as pd
import numpy as np
import seaborn as sea
import matplotlib.pyplot as plt

import sys
sys.path.append('..')

from src.eda import reporte_calidad
from src.eda import clasificar_formato_fecha
from src.eda import resolver_duplicados
from src.eda import validar_columna_temporal
from src.eda import orquestar_transformaciones_cols

## 1) FACT_MANTENIMIENTO

In [114]:
directory =  r'C:\Users\jbard\Desktop\Library\@CURSOS ACTUALES\@Data Analitycs\Python projects\Mineria-simulacion v2'
fact_mtto_PC= pd.read_csv(directory+r'\data\raw\fact_mantenimiento_PC_raw.csv')
fact_mtto_PN= pd.read_csv(directory+r'\data\raw\fact_mantenimiento_PN_raw.csv')
fact_mtto_PS= pd.read_csv(directory+r'\data\raw\fact_mantenimiento_PS_raw.csv')


#### 1) FACT_MANTENIMIENTO_PC

1era verificacion de calidad

In [115]:
from src.eda import reporte_calidad
reporte_calidad(fact_mtto_PC,nombre_df='fact_mtto_PC')




--- Resumen: fact_mtto_PC ---
Total filas del df: 2735
Sumatoria filas duplicadas: 27 (0.99%)
Columnas: 9


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,repuesto_principal,str,0.009872,2684,51,1.86,7
1,orden_id,str,0.009872,2735,0,0.00,2708
2,parada_id,str,0.009872,2735,0,0.00,2708
3,fecha_apertura,str,0.009872,2735,0,0.00,1271
4,equipo_id,str,0.009872,2735,0,0.00,10
5,tipo_orden,str,0.009872,2735,0,0.00,2
6,horas_hombre,float64,0.009872,2735,0,0.00,306
7,num_tecnicos,int64,0.009872,2735,0,0.00,3
8,costo_repuestos_soles,float64,0.009872,2735,0,0.00,2233


VERIFICACION TIPOS DE FECHA QUE HAY EN LA COLUMNA 'FECHA_APERTURA'

In [116]:
tipos_fecha=pd.DataFrame(
    {
        'fecha_apertura': fact_mtto_PC['fecha_apertura'],
        'tipo_fecha': fact_mtto_PC['fecha_apertura'].apply(clasificar_formato_fecha)
    }
)
display(tipos_fecha.head(3))
tipos_de_fecha= tipos_fecha['tipo_fecha'].value_counts()
tipos_de_fecha

#Como no existen mas de otros formatos de fecha, por tanto podemos aplicar directamente la trasnformacion. Hay que tomar en cuenta que quiza haya mas de un formato de fecha en la columna, si ese fuera el caso esto generaria que tengamos hacer tranformacion independiente y un simple pd.to_datetime(df['columna']) no seria suficiente.
#LA SOLUCION sera crear una pipeline para cada escenario o por el contrario hacerlo manualmente para cada tipo_fecha diferente. EN ESTA OCASION EL PIPELINE DE LA VALIDACION DE FECHA YA ESTA REALIZADO DENTRO DE EDA.PY

,fecha_apertura,tipo_fecha
0,2024-05-28,YMD
1,2027-08-24,YMD
2,2027-09-11,YMD


tipo_fecha
YMD    2735
Name: count, dtype: int64

In [117]:
tipos_fecha=pd.DataFrame(
    {
        'fecha_apertura': fact_mtto_PC['fecha_apertura'],
        'tipo_fecha': fact_mtto_PC['fecha_apertura'].apply(clasificar_formato_fecha)
    }
)
display(tipos_fecha.head(3))
tipos_de_fecha= tipos_fecha['tipo_fecha'].value_counts()
tipos_de_fecha

validar_columna_temporal(fact_mtto_PC,'fecha_apertura',tipo='fecha')

,fecha_apertura,tipo_fecha
0,2024-05-28,YMD
1,2027-08-24,YMD
2,2027-09-11,YMD


  ↳ [validar_columna_temporal] Verificación de 'fecha_apertura' (tipo=fecha):
YMD    2735
Name: count, dtype: int64

  ↳ [validar_columna_temporal] Validación correcta. Formato único. Columna convertida.



,orden_id,parada_id,equipo_id,fecha_apertura,tipo_orden,horas_hombre,num_tecnicos,repuesto_principal,costo_repuestos_soles
0,OT-004314,PAR-005657,PC-EQ-06,2024-05-28,Correctiva,12.7,3,Faja/correa,493.0
1,OT-005483,PAR-007230,PC-EQ-10,2027-08-24,Correctiva,13.1,2,Ninguno (solo mano de obra),4001.0
2,OT-003042,PAR-004020,PC-EQ-01,2027-09-11,Preventiva,9.7,3,Ninguno (solo mano de obra),1002.0
3,OT-005094,PAR-006708,PC-EQ-09,2025-10-03,Correctiva,0.6,2,Ninguno (solo mano de obra),5051.0
4,OT-004318,PAR-005661,PC-EQ-06,2024-06-06,Correctiva,1.0,2,NaN,5757.0
...,...,...,...,...,...,...,...,...,...
2730,OT-003349,PAR-004424,PC-EQ-03,2024-03-20,Correctiva,0.9,2,Sello mecánico,7199.0
2731,OT-002985,PAR-003939,PC-EQ-01,2026-11-24,Correctiva,3.7,2,Sensor/instrumento,2870.0
2732,OT-004610,PAR-006038,PC-EQ-07,2025-03-24,Correctiva,8.7,2,Sello mecánico,385.0
2733,OT-003238,PAR-004289,PC-EQ-02,2026-09-06,Correctiva,2.2,2,Motor eléctrico,5208.0


In [118]:

if fact_mtto_PN['repuesto_principal'].isnull().any():
    fact_mtto_PN['repuesto_principal'] = fact_mtto_PN['repuesto_principal'].fillna('indeterminado')# Partamos del escenario donde algunos repuestos se perdería el nombre, sin embargo los costos no, por tanto simplemente averiguariamos el nombre del repuesto y reemplazariamos luego. Si existen nulos iría por defecto bajo el nombre 'Indeterminado'

    print("Nulos encontrados y reemplazados por 'indeterminado'.")
else:
    print("No se encontraron nulos en 'repuesto_principal', no se aplica transformación.")




Nulos encontrados y reemplazados por 'indeterminado'.


#### 2) FACT_MANTENIMIENTO_PS

In [119]:
from src.eda import reporte_calidad
reporte_calidad(fact_mtto_PS,nombre_df='fact_mtto_PS')




--- Resumen: fact_mtto_PS ---
Total filas del df: 2422
Sumatoria filas duplicadas: 23 (0.95%)
Columnas: 9


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,repuesto_principal,str,0.009496,2376,46,1.9,7
1,orden_id,str,0.009496,2422,0,0.0,2399
2,parada_id,str,0.009496,2422,0,0.0,2399
3,fecha_apertura,str,0.009496,2422,0,0.0,1204
4,equipo_id,str,0.009496,2422,0,0.0,10
5,tipo_orden,str,0.009496,2422,0,0.0,2
6,horas_hombre,float64,0.009496,2422,0,0.0,277
7,num_tecnicos,int64,0.009496,2422,0,0.0,3
8,costo_repuestos_soles,float64,0.009496,2422,0,0.0,1999


In [120]:

if fact_mtto_PS['repuesto_principal'].isnull().any():
    fact_mtto_PS['repuesto_principal'] = fact_mtto_PS['repuesto_principal'].fillna('indeterminado')# Partamos del escenario donde algunos repuestos se perdería el nombre, sin embargo los costos no, por tanto simplemente averiguariamos el nombre del repuesto y reemplazariamos luego. Si existen nulos iría por defecto bajo el nombre 'Indeterminado'

    print("Nulos encontrados en 'repuesto_principal' y reemplazados por 'indeterminado'.")
else:
    print("No se encontraron nulos en 'repuesto_principal', no se aplica transformación.")




Nulos encontrados en 'repuesto_principal' y reemplazados por 'indeterminado'.


In [121]:
tipos_fecha=pd.DataFrame(
    {
        'fecha_apertura': fact_mtto_PS['fecha_apertura'],
        'tipo_fecha': fact_mtto_PS['fecha_apertura'].apply(clasificar_formato_fecha)
    }
)
display(tipos_fecha.head(3))
tipos_de_fecha= tipos_fecha['tipo_fecha'].value_counts()
tipos_de_fecha

#Como no existen mas de otros formatos de fecha, por tanto podemos aplicar directamente la trasnformacion. Hay que tomar en cuenta que quiza haya mas de un formato de fecha en la columna, si ese fuera el caso esto generaria que tengamos hacer tranformacion independiente y un simple pd.to_datetime(df['columna']) no seria suficiente.
#LA SOLUCION sera crear una pipeline para cada escenario o por el contrario hacerlo manualmente para cada tipo_fecha diferente. EN ESTA OCASION EL PIPELINE DE LA VALIDACION DE FECHA YA ESTA REALIZADO DENTRO DE EDA.PY


,fecha_apertura,tipo_fecha
0,2024-10-14,YMD
1,2025-06-24,YMD
2,2024-09-17,YMD


tipo_fecha
YMD    2422
Name: count, dtype: int64

In [122]:
validar_columna_temporal(fact_mtto_PS,'fecha_apertura',tipo='fecha')

  ↳ [validar_columna_temporal] Verificación de 'fecha_apertura' (tipo=fecha):
YMD    2422
Name: count, dtype: int64

  ↳ [validar_columna_temporal] Validación correcta. Formato único. Columna convertida.



,orden_id,parada_id,equipo_id,fecha_apertura,tipo_orden,horas_hombre,num_tecnicos,repuesto_principal,costo_repuestos_soles
0,OT-006852,PAR-009048,PS-EQ-06,2024-10-14,Preventiva,3.8,3,Sensor/instrumento,1951.0
1,OT-006899,PAR-009107,PS-EQ-06,2025-06-24,Correctiva,1.8,1,Rodamiento,5263.0
2,OT-007564,PAR-010013,PS-EQ-09,2024-09-17,Correctiva,10.5,2,Sello mecánico,4010.0
3,OT-006307,PAR-008321,PS-EQ-04,2024-12-29,Correctiva,1.0,2,Sensor/instrumento,2647.0
4,OT-005711,PAR-007532,PS-EQ-01,2027-01-31,Preventiva,4.4,3,Rodamiento,2069.0
...,...,...,...,...,...,...,...,...,...
2417,OT-007068,PAR-009326,PS-EQ-06,2027-11-15,Correctiva,14.7,3,Ninguno (solo mano de obra),1804.0
2418,OT-005951,PAR-007840,PS-EQ-02,2026-08-01,Correctiva,4.3,3,Ninguno (solo mano de obra),872.0
2419,OT-005653,PAR-007450,PS-EQ-01,2026-03-28,Preventiva,5.5,2,Faja/correa,744.0
2420,OT-007014,PAR-009255,PS-EQ-06,2027-01-30,Preventiva,9.1,2,Ninguno (solo mano de obra),2173.0


#### 3) FACT_MANTENIMIENTO_PN

In [123]:
from src.eda import reporte_calidad
reporte_calidad(fact_mtto_PN,nombre_df='fact_mtto_PN')


--- Resumen: fact_mtto_PN ---
Total filas del df: 2829
Sumatoria filas duplicadas: 28 (0.99%)
Columnas: 9


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,orden_id,str,0.009897,2829,0,0.0,2801
1,parada_id,str,0.009897,2829,0,0.0,2801
2,equipo_id,str,0.009897,2829,0,0.0,10
3,fecha_apertura,str,0.009897,2829,0,0.0,1273
4,tipo_orden,str,0.009897,2829,0,0.0,2
5,horas_hombre,float64,0.009897,2829,0,0.0,299
6,num_tecnicos,int64,0.009897,2829,0,0.0,3
7,repuesto_principal,str,0.009897,2829,0,0.0,8
8,costo_repuestos_soles,float64,0.009897,2829,0,0.0,2310


In [124]:
# Descartado 1, no puedo depender de verificar cada una de las tablas si es que se detectaran duplicados, la solucion es diseñar un pipeline que me permita identificar que hacer en caso se detecten duplicados, cuando considerar duplicados y cuando no.... todo manejado dentro de una funcion que pueda reutilizar siempre que se detecten duplicados
fact_mtto_PN[fact_mtto_PN.duplicated()].sort_values('orden_id')

,orden_id,parada_id,equipo_id,fecha_apertura,tipo_orden,horas_hombre,num_tecnicos,repuesto_principal,costo_repuestos_soles
2316,OT-000214,PAR-000282,PN-EQ-01,2027-03-19,Preventiva,7.7,3,Faja/correa,1274.0
1856,OT-000312,PAR-000425,PN-EQ-02,2024-09-26,Correctiva,9.7,1,Rodamiento,1420.0
952,OT-000486,PAR-000648,PN-EQ-02,2027-02-21,Correctiva,19.4,1,Sello mecánico,5666.0
2278,OT-000631,PAR-000852,PN-EQ-03,2025-12-28,Preventiva,4.7,3,Rodamiento,1921.0
2092,OT-000705,PAR-000961,PN-EQ-03,2027-05-18,Preventiva,7.1,3,Revestimiento (liner),2316.0
2770,OT-000966,PAR-001282,PN-EQ-04,2026-05-06,Preventiva,6.2,3,Motor eléctrico,536.0
2081,OT-000996,PAR-001322,PN-EQ-04,2026-09-21,Correctiva,8.0,1,Motor eléctrico,186.0
1957,OT-001146,PAR-001506,PN-EQ-05,2024-05-31,Correctiva,19.4,2,Sensor/instrumento,7635.0
2181,OT-001197,PAR-001571,PN-EQ-05,2025-02-04,Correctiva,14.5,1,Sensor/instrumento,3544.0
2774,OT-001259,PAR-001650,PN-EQ-05,2025-10-11,Preventiva,6.6,3,Ninguno (solo mano de obra),2077.0


In [125]:

if fact_mtto_PN['repuesto_principal'].isnull().any():
    fact_mtto_PN['repuesto_principal'] = fact_mtto_PN['repuesto_principal'].fillna('indeterminado')# Partamos del escenario donde algunos repuestos se perdería el nombre, sin embargo los costos no, por tanto simplemente averiguariamos el nombre del repuesto y reemplazariamos luego. Si existen nulos iría por defecto bajo el nombre 'Indeterminado'

    print("Nulos encontrados en 'repuesto_principal' y reemplazados por 'indeterminado'.")
else:
    print("No se encontraron nulos en 'repuesto_principal', no se aplica transformación.")




No se encontraron nulos en 'repuesto_principal', no se aplica transformación.


In [126]:
tipos_fecha=pd.DataFrame(
    {
        'fecha_apertura': fact_mtto_PN['fecha_apertura'],
        'tipo_fecha': fact_mtto_PN['fecha_apertura'].apply(clasificar_formato_fecha)
    }
)
display(tipos_fecha.head(3))
tipos_de_fecha= tipos_fecha['tipo_fecha'].value_counts()
tipos_de_fecha

#Como no existen mas de otros formatos de fecha, por tanto podemos aplicar directamente la trasnformacion. Hay que tomar en cuenta que quiza haya mas de un formato de fecha en la columna, si ese fuera el caso esto generaria que tengamos hacer tranformacion independiente y un simple pd.to_datetime(df['columna']) no seria suficiente.
#LA SOLUCION sera crear una pipeline para cada escenario o por el contrario hacerlo manualmente para cada tipo_fecha diferente. EN ESTA OCASION EL PIPELINE DE LA VALIDACION DE FECHA YA ESTA REALIZADO DENTRO DE EDA.PY

,fecha_apertura,tipo_fecha
0,2024-09-23,YMD
1,2026-03-06,YMD
2,2027-09-29,YMD


tipo_fecha
YMD    2829
Name: count, dtype: int64

In [127]:
reporte_calidad(fact_mtto_PN)


--- Resumen: DataFrame ---
Total filas del df: 2829
Sumatoria filas duplicadas: 28 (0.99%)
Columnas: 9


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,orden_id,str,0.009897,2829,0,0.0,2801
1,parada_id,str,0.009897,2829,0,0.0,2801
2,equipo_id,str,0.009897,2829,0,0.0,10
3,fecha_apertura,str,0.009897,2829,0,0.0,1273
4,tipo_orden,str,0.009897,2829,0,0.0,2
5,horas_hombre,float64,0.009897,2829,0,0.0,299
6,num_tecnicos,int64,0.009897,2829,0,0.0,3
7,repuesto_principal,str,0.009897,2829,0,0.0,8
8,costo_repuestos_soles,float64,0.009897,2829,0,0.0,2310


#### A) RESOLUCION MEDIANTE ITERACION PARA  DUPLICADOS USANDO UNA FUNCION REHUTILIZABLE 

In [128]:
tablas_mantenimiento = {
    "PN": fact_mtto_PN,
    "PC": fact_mtto_PC,
    "PS": fact_mtto_PS,
}

limpio_mtto = {} #diccionario de dataframes limpios
exactos_mtto = {} #diccionario de dataframes exactos
conflictos_mtto = {} #diccionario de dataframes conflictivos

for planta, df in tablas_mantenimiento.items():
    limpio, exactos, conflictos = resolver_duplicados( #Los 3 outputs del return de la funcion resolver duplicados
        df,
        subset_clave=['orden_id', 'equipo_id','fecha_apertura'], # El evento es el orden_equipo_fecha )la agrupacion de ellos es el evento
        columnas_grano_esperado=['parada_id','tipo_orden'], # Solo puede haber existir de forma unica el contenido del grano, es la unica q manda alerta de grano roto
        nombre_tabla=f"Mantenimiento {planta}"
    ) #Error cometido, una orden no puede tener mas de una fecha de apertura, 'fecha_apertura' es una parte del evento unico del grano

    # Aplicando ordenamiento de las ordenes antes de mostrarlas
    limpio = limpio.sort_values('orden_id')
    exactos = exactos.sort_values('orden_id')
    conflictos = conflictos.sort_values('orden_id')

    # Mostrando resultados por cada planta(usar kEYS no values)
    limpio_mtto[planta] = limpio
    exactos_mtto[planta] = exactos
    conflictos_mtto[planta] = conflictos

    

[Mantenimiento PN] duplicados exactos: 56 filas | con conflicto real: 0 filas
[Mantenimiento PC] duplicados exactos: 54 filas | con conflicto real: 0 filas
[Mantenimiento PS] duplicados exactos: 46 filas | con conflicto real: 0 filas


#### B) Verificacion manual DE RESOLVER_DUPLICADOS, SI ES CONVENIENTE

In [129]:
# (DESPLEGAR) Proceda a verificar si encontrol conflictos, o busca verificar cuales elementos exactos son los duplicados
#PLANTA NORTE
limpio_mtto['PN']
exactos_mtto['PN']
conflictos_mtto['PN']

#PLANTA SUR


#PLANTA CENTRO


,orden_id,parada_id,equipo_id,fecha_apertura,tipo_orden,horas_hombre,num_tecnicos,repuesto_principal,costo_repuestos_soles


#### C) VERIFICACION FINAL DE INTEGRIDAD DE DATOS Y CONCATENACION DE DATAFRAMES

In [130]:
# 1) Agregar trazabilidad de planta
for planta, df in limpio_mtto.items(): #Tomamos solo lo limpio que se guardo en un diccionario de dataframes limpios
    df['planta'] = planta #asignar nombre de planta como nueva columna

# 2) Verificación 3/3 completa (duplicados + conflictos en los 3 dataframes)
duplicados_por_planta = {p: df.duplicated().sum() for p, df in limpio_mtto.items()}
conflictos_por_planta = {p: len(df) for p, df in conflictos_mtto.items()}

#Cumple las 2 condiciones para ser TRUE
todo_ok = all(v == 0 for v in duplicados_por_planta.values()) and \
          all(v == 0 for v in conflictos_por_planta.values())

print("Duplicados:", duplicados_por_planta)
print("Conflictos:", conflictos_por_planta)

if todo_ok:
    print("\n,✅ 3/3 plantas limpias y sin conflictos, se procede a concatenar")
    fact_mtto_final = pd.concat(limpio_mtto.values(), ignore_index=True) #pierden su indice original --> un nuevo indice al concatenado
    
    # 3) Chequeo de duplicados cruzados entre plantas
    dup_cruzados = fact_mtto_final.duplicated(subset=['orden_id','equipo_id']).sum() #Se verifica duplicados sobre el concat_df, se suma y se muestra
    print(f"Duplicados cruzados entre plantas: {dup_cruzados}")
    
    print(r'Shape del dataframe',fact_mtto_final.shape,'Mostrando head del dataframe:')
    display(fact_mtto_final.head())
else:
    print("❌ Revisar antes de concatenar")

# 4) Checkpoint final: el conteo por planta debe coincidir con el tamaño de limpio_mtto original
conteo_esperado = {p: len(df) for p, df in limpio_mtto.items()} #diccionario: cantidad del df limpio, en el diccionario de los df limpios
conteo_real = fact_mtto_final['planta'].value_counts().to_dict() # value_counts a las plantas del df concatenado

print('Checkpint de conteo_esperado VS conteo_real:')
print("Esperado:", conteo_esperado)
print("Real:", conteo_real)

assert conteo_esperado == conteo_real, "❌ El conteo no coincide, revisar antes de exportar"
print("✅ Conteo coincide, listo para exportar")

Duplicados: {'PN': np.int64(0), 'PC': np.int64(0), 'PS': np.int64(0)}
Conflictos: {'PN': 0, 'PC': 0, 'PS': 0}

,✅ 3/3 plantas limpias y sin conflictos, se procede a concatenar
Duplicados cruzados entre plantas: 0
Shape del dataframe (7908, 10) Mostrando head del dataframe:


,orden_id,parada_id,equipo_id,fecha_apertura,tipo_orden,horas_hombre,num_tecnicos,repuesto_principal,costo_repuestos_soles,planta
0,OT-000001,PAR-000001,PN-EQ-01,2024-01-09,Correctiva,3.3,1,Rodamiento,6676.0,PN
1,OT-000002,PAR-000002,PN-EQ-01,2024-01-15,Preventiva,4.0,1,Ninguno (solo mano de obra),267.0,PN
2,OT-000003,PAR-000003,PN-EQ-01,2024-01-19,Correctiva,5.2,1,Ninguno (solo mano de obra),4601.0,PN
3,OT-000004,PAR-000004,PN-EQ-01,2024-01-25,Correctiva,13.1,3,Faja/correa,4.0,PN
4,OT-000005,PAR-000006,PN-EQ-01,2024-01-29,Preventiva,7.3,2,Sello mecánico,773.0,PN


Checkpint de conteo_esperado VS conteo_real:
Esperado: {'PN': 2801, 'PC': 2708, 'PS': 2399}
Real: {'PN': 2801, 'PC': 2708, 'PS': 2399}
✅ Conteo coincide, listo para exportar


### D) EXPORTACION Fact_mtto_ limpiado_conatenado_multiplanta Y CREACION DE LOG

In [131]:
# import os
# from datetime import datetime

# ruta_salida = r"C:\Users\jbard\Desktop\Library\@CURSOS ACTUALES\@Data Analitycs\Python projects\Mineria-simulacion v2\data\processed/fact_mantenimiento.csv"
# ruta_log    = r"C:\Users\jbard\Desktop\Library\@CURSOS ACTUALES\@Data Analitycs\Python projects\Mineria-simulacion v2\data\processed/_log_exportaciones.csv"

# fact_mtto_final.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

# registro = pd.DataFrame([{
#     "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#     "archivo": "fact_mantenimiento.csv",
#     "filas": len(fact_mtto_final),
#     "columnas": len(fact_mtto_final.columns),
#     "duplicados_por_planta": str(duplicados_por_planta),
#     "conflictos_por_planta": str(conflictos_por_planta),
# }])

# registro.to_csv(ruta_log, mode="a", index=False,
#                  header=not os.path.exists(ruta_log), encoding="utf-8-sig")

# print(f"✅ Exportado: {ruta_salida}")
# print(f"📋 Log actualizado: {ruta_log}")

## 2) FACT_PARADAS

In [132]:
directory =  r'C:\Users\jbard\Desktop\Library\@CURSOS ACTUALES\@Data Analitycs\Python projects\Mineria-simulacion v2'
fact_paradas_PC= pd.read_csv(directory+r'\data\raw\fact_paradas_PC_raw.csv')
fact_paradas_PN= pd.read_csv(directory+r'\data\raw\fact_paradas_PN_raw.csv')
fact_paradas_PS= pd.read_csv(directory+r'\data\raw\fact_paradas_PS_raw.csv')


In [133]:
#REVISION DE CALIDAD PRIMARIA
fact_paradas_dicc = {
    "PN": fact_paradas_PN,
    "PC": fact_paradas_PC,
    "PS": fact_paradas_PS
}

for planta, df in fact_paradas_dicc.items():
    display(reporte_calidad(df, nombre_df=f"fact_paradas_{planta}"))




--- Resumen: fact_paradas_PN ---
Total filas del df: 3756
Sumatoria filas duplicadas: 73 (1.94%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,causa,str,0.019436,3580,176,4.69,16
1,costo_estimado_soles,float64,0.019436,3617,139,3.70,3071
2,parada_id,str,0.019436,3756,0,0.00,3683
3,equipo_id,str,0.019436,3756,0,0.00,70
4,hora_inicio_aprox,str,0.019436,3756,0,0.00,23
5,fecha,str,0.019436,3756,0,0.00,2078
6,tipo_parada,str,0.019436,3756,0,0.00,12
7,duracion_horas,float64,0.019436,3756,0,0.00,201



--- Resumen: fact_paradas_PC ---
Total filas del df: 3650
Sumatoria filas duplicadas: 71 (1.95%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,causa,str,0.019452,3481,169,4.63,16
1,costo_estimado_soles,float64,0.019452,3503,147,4.03,2980
2,parada_id,str,0.019452,3650,0,0.00,3579
3,equipo_id,str,0.019452,3650,0,0.00,70
4,hora_inicio_aprox,str,0.019452,3650,0,0.00,23
5,fecha,str,0.019452,3650,0,0.00,2039
6,tipo_parada,str,0.019452,3650,0,0.00,12
7,duracion_horas,float64,0.019452,3650,0,0.00,210



--- Resumen: fact_paradas_PS ---
Total filas del df: 3330
Sumatoria filas duplicadas: 65 (1.95%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,causa,str,0.01952,3169,161,4.83,16
1,costo_estimado_soles,float64,0.01952,3203,127,3.81,2705
2,parada_id,str,0.01952,3330,0,0.00,3265
3,equipo_id,str,0.01952,3330,0,0.00,70
4,hora_inicio_aprox,str,0.01952,3330,0,0.00,23
5,fecha,str,0.01952,3330,0,0.00,1972
6,tipo_parada,str,0.01952,3330,0,0.00,12
7,duracion_horas,float64,0.01952,3330,0,0.00,202


Tienen errores en los mismos lugares, se podría suponer que alguno de los formatos de llenado esta mal o algo en los reportes hace que se genere string a las fechas, pero de todas formas al ser consistente en las mismas columnas, podemos aplicar trasnformaciones en masa facilmente a las 3 columnas de fechas en string a la vez

In [134]:
#Se resuelven duplicados, se aplica a todas las tablas de paradas mediante iteación
fact_paradas_dicc = {
    "PN": fact_paradas_PN,
    "PC": fact_paradas_PC,
    "PS": fact_paradas_PS
}

fact_paradas_limpiado = {} # Diccionario vacio que va recibir la key y el value(df.name_limpiado)
exactos_paradas_limpiado = {}
conflictos_paradas_limpiado ={} #usas la clave para "entrar" al diccionario y sacar el valor necesario (df)

for planta, df in fact_paradas_dicc.items():
    limpio,exactos,conflictos=resolver_duplicados(
        df, subset_clave=['parada_id','equipo_id'],
        columnas_grano_esperado=['fecha','tipo_parada','fecha','hora_inicio_aprox'])
        
    fact_paradas_limpiado[planta]=limpio
    exactos_paradas_limpiado[planta]=exactos
    conflictos_paradas_limpiado[planta]=conflictos

#Verificacion de calidad del dataframe limpiado
for planta, df in fact_paradas_limpiado.items():
    display(reporte_calidad(df, nombre_df=f"fact_paradas_{planta}"))

# Para verificar construimos bien nuestros diccionarios
# fact_paradas_PN_limpiado.keys() 
# fact_paradas_PC_limpiado.keys() 

[] duplicados exactos: 146 filas | con conflicto real: 0 filas
[] duplicados exactos: 142 filas | con conflicto real: 0 filas
[] duplicados exactos: 130 filas | con conflicto real: 0 filas

--- Resumen: fact_paradas_PN ---
Total filas del df: 3683
Sumatoria filas duplicadas: 0 (0.00%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,causa,str,0.0,3510,173,4.70,16
1,costo_estimado_soles,float64,0.0,3545,138,3.75,3071
2,parada_id,str,0.0,3683,0,0.00,3683
3,equipo_id,str,0.0,3683,0,0.00,70
4,hora_inicio_aprox,str,0.0,3683,0,0.00,23
5,fecha,str,0.0,3683,0,0.00,2078
6,tipo_parada,str,0.0,3683,0,0.00,12
7,duracion_horas,float64,0.0,3683,0,0.00,201



--- Resumen: fact_paradas_PC ---
Total filas del df: 3579
Sumatoria filas duplicadas: 0 (0.00%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,causa,str,0.0,3413,166,4.64,16
1,costo_estimado_soles,float64,0.0,3433,146,4.08,2980
2,parada_id,str,0.0,3579,0,0.00,3579
3,equipo_id,str,0.0,3579,0,0.00,70
4,hora_inicio_aprox,str,0.0,3579,0,0.00,23
5,fecha,str,0.0,3579,0,0.00,2039
6,tipo_parada,str,0.0,3579,0,0.00,12
7,duracion_horas,float64,0.0,3579,0,0.00,210



--- Resumen: fact_paradas_PS ---
Total filas del df: 3265
Sumatoria filas duplicadas: 0 (0.00%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,causa,str,0.0,3107,158,4.84,16
1,costo_estimado_soles,float64,0.0,3138,127,3.89,2705
2,parada_id,str,0.0,3265,0,0.00,3265
3,equipo_id,str,0.0,3265,0,0.00,70
4,hora_inicio_aprox,str,0.0,3265,0,0.00,23
5,fecha,str,0.0,3265,0,0.00,1972
6,tipo_parada,str,0.0,3265,0,0.00,12
7,duracion_horas,float64,0.0,3265,0,0.00,202


In [135]:
# ============================================================
# 1) CONTRATO: qué tipo de dato espero en cada columna (a mano)
# ============================================================
esquema_paradas = {
    'fecha':                 'fecha',
    'hora_inicio_aprox':     'hora',
    'costo_estimado_soles':  'decimal',
    'causa':                 'categorico',
    'duracion_horas':        'decimal',
    'tipo_parada':           'categorico',
    'equipo_id':             'categorico',
}
fact_paradas_PC = orquestar_transformaciones_cols(fact_paradas_PC, esquema_paradas)
fact_paradas_PS = orquestar_transformaciones_cols(fact_paradas_PS, esquema_paradas)
fact_paradas_PN = orquestar_transformaciones_cols(fact_paradas_PN, esquema_paradas)


[orquestador] CONVIERTE 'fecha': str -> fecha (esperado datetime64[us])
  ↳ [validar_columna_temporal] Verificación de 'fecha' (tipo=fecha):
YMD          2246
DMY_SLASH    1404
Name: count, dtype: int64

  ↳ [validar_columna_temporal] La columna NO pasó la validación de formato único.
  ↳ [validar_columna_temporal] Formatos encontrados: {'YMD': np.int64(2246), 'DMY_SLASH': np.int64(1404)}
  ↳ [validar_columna_temporal] Por consecuente, aplicando conversión por máscara (cada subconjunto con su propio formato para cada tipo detectado)...
  ↳ [validar_columna_temporal] Conversión por máscara aplicada: 3650/3650 filas convertidas correctamente.


[orquestador] CONVIERTE 'hora_inicio_aprox': str -> hora (esperado <class 'object'>)
  ↳ [validar_columna_temporal] Verificación de 'hora_inicio_aprox' (tipo=hora):
HH_MM    3650
Name: count, dtype: int64

  ↳ [validar_columna_temporal] Validación correcta. Formato único. Columna convertida.

[orquestador] OK 'costo_estimado_soles': ya es decimal

In [136]:
# diccionario de entrada: una planta -> su DataFrame ya con dtypes correctos
fact_paradas = {
    'PN': fact_paradas_PN,
    'PC': fact_paradas_PC,
    'PS': fact_paradas_PS,
}

# diccionarios de salida: una entrada por planta en cada uno
fact_paradas_limpios = {}
fact_paradas_exactos = {}
fact_paradas_conflictivos = {}

for planta_id, df_planta in fact_paradas.items():
    limpio, exactos, conflictivos = resolver_duplicados(
        df_planta,
        subset_clave=['parada_id'],
        columnas_desempate=None,
        nombre_tabla=f"fact_paradas_{planta_id}",
        columnas_grano_esperado=[
            'equipo_id', 'fecha', 'hora_inicio_aprox',
            'duracion_horas', 'tipo_parada', 'causa', 'costo_estimado_soles'
        ]
    )
    fact_paradas_limpios[planta_id] = limpio
    fact_paradas_exactos[planta_id] = exactos
    fact_paradas_conflictivos[planta_id] = conflictivos



[fact_paradas_PN] duplicados exactos: 146 filas | con conflicto real: 0 filas
[fact_paradas_PC] duplicados exactos: 142 filas | con conflicto real: 0 filas
[fact_paradas_PS] duplicados exactos: 130 filas | con conflicto real: 0 filas


In [137]:
#Verificacion de calidad del dataframe limpiado
for planta, df in fact_paradas_limpios.items():
    display(reporte_calidad(df, nombre_df=f"fact_paradas_{planta}"))


--- Resumen: fact_paradas_PN ---
Total filas del df: 3683
Sumatoria filas duplicadas: 0 (0.00%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,causa,str,0.0,3510,173,4.70,16
1,costo_estimado_soles,float64,0.0,3545,138,3.75,3071
2,parada_id,str,0.0,3683,0,0.00,3683
3,equipo_id,str,0.0,3683,0,0.00,70
4,hora_inicio_aprox,object,0.0,3683,0,0.00,23
5,fecha,datetime64[us],0.0,3683,0,0.00,1371
6,tipo_parada,str,0.0,3683,0,0.00,12
7,duracion_horas,float64,0.0,3683,0,0.00,201



--- Resumen: fact_paradas_PC ---
Total filas del df: 3579
Sumatoria filas duplicadas: 0 (0.00%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,causa,str,0.0,3413,166,4.64,16
1,costo_estimado_soles,float64,0.0,3433,146,4.08,2980
2,parada_id,str,0.0,3579,0,0.00,3579
3,equipo_id,str,0.0,3579,0,0.00,70
4,hora_inicio_aprox,object,0.0,3579,0,0.00,23
5,fecha,datetime64[us],0.0,3579,0,0.00,1349
6,tipo_parada,str,0.0,3579,0,0.00,12
7,duracion_horas,float64,0.0,3579,0,0.00,210



--- Resumen: fact_paradas_PS ---
Total filas del df: 3265
Sumatoria filas duplicadas: 0 (0.00%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,causa,str,0.0,3107,158,4.84,16
1,costo_estimado_soles,float64,0.0,3138,127,3.89,2705
2,parada_id,str,0.0,3265,0,0.00,3265
3,equipo_id,str,0.0,3265,0,0.00,70
4,hora_inicio_aprox,object,0.0,3265,0,0.00,23
5,fecha,datetime64[us],0.0,3265,0,0.00,1326
6,tipo_parada,str,0.0,3265,0,0.00,12
7,duracion_horas,float64,0.0,3265,0,0.00,202


In [138]:
# 1) Agregar trazabilidad de planta
for planta, df in fact_paradas_limpios.items():
    df['planta'] = planta

# 2) Verificación 3/3 completa (duplicados + conflictos en los 3 dataframes)
duplicados_por_planta_paradas = {p: df.duplicated().sum() for p, df in fact_paradas_limpios.items()}
conflictos_por_planta_paradas = {p: len(df) for p, df in fact_paradas_conflictivos.items()}

todo_ok_paradas = all(v == 0 for v in duplicados_por_planta_paradas.values()) and \
                   all(v == 0 for v in conflictos_por_planta_paradas.values())

print("Duplicados:", duplicados_por_planta_paradas)
print("Conflictos:", conflictos_por_planta_paradas)

if not todo_ok_paradas:
    raise RuntimeError(
        f"❌ Revisar antes de concatenar. Duplicados: {duplicados_por_planta_paradas} | "
        f"Conflictos: {conflictos_por_planta_paradas}"
    )

print("\n✅ 3/3 plantas limpias y sin conflictos, se procede a concatenar")
fact_paradas_final = pd.concat(fact_paradas_limpios.values(), ignore_index=True)

# 3) Chequeo de duplicados cruzados entre plantas (parada_id es la clave única real de esta tabla)
dup_cruzados_paradas = fact_paradas_final.duplicated(subset=['parada_id']).sum()
if dup_cruzados_paradas > 0:
    raise RuntimeError(f"❌ {dup_cruzados_paradas} duplicados cruzados entre plantas -- revisar antes de exportar")
print(f"Duplicados cruzados entre plantas: {dup_cruzados_paradas}")

print('Shape del dataframe', fact_paradas_final.shape, 'Mostrando head del dataframe:')
display(fact_paradas_final.head())

# 4) Checkpoint final: el conteo por planta debe coincidir con el tamaño de fact_paradas_limpios original
conteo_esperado_paradas = {p: len(df) for p, df in fact_paradas_limpios.items()}
conteo_real_paradas = fact_paradas_final['planta'].value_counts().to_dict()

print('Checkpoint de conteo_esperado VS conteo_real:')
print("Esperado:", conteo_esperado_paradas)
print("Real:", conteo_real_paradas)

assert conteo_esperado_paradas == conteo_real_paradas, "❌ El conteo no coincide, revisar antes de exportar"
print("✅ Conteo coincide, listo para exportar")

Duplicados: {'PN': np.int64(0), 'PC': np.int64(0), 'PS': np.int64(0)}
Conflictos: {'PN': 0, 'PC': 0, 'PS': 0}

✅ 3/3 plantas limpias y sin conflictos, se procede a concatenar
Duplicados cruzados entre plantas: 0
Shape del dataframe (10527, 9) Mostrando head del dataframe:


,parada_id,equipo_id,fecha,hora_inicio_aprox,duracion_horas,tipo_parada,causa,costo_estimado_soles,planta
0,PAR-001518,PN-EQ-05,2024-07-21,01:00:00,8.5,preventiva,Mantenimiento programado (plan anual),8271.0,PN
1,PAR-001890,PN-EQ-05,2027-11-29,15:00:00,4.7,preventiva,Mantenimiento programado (plan anual),8733.0,PN
2,PAR-000180,PN-EQ-01,2025-12-07,14:00:00,8.5,Preventiva,Mantenimiento programado (plan anual),18203.0,PN
3,PAR-000501,PNEQ02,2025-07-20,18:00:00,0.9,Operativa,Corte de energía externo,444.0,PN
4,PAR-000025,pn-eq-01,2024-04-04,15:00:00,4.6,Correctiva,Falla mecánica - desgaste de componente,5492.0,PN


Checkpoint de conteo_esperado VS conteo_real:
Esperado: {'PN': 3683, 'PC': 3579, 'PS': 3265}
Real: {'PN': 3683, 'PC': 3579, 'PS': 3265}
✅ Conteo coincide, listo para exportar


In [139]:
import os
from datetime import datetime

ruta_salida = r"C:\Users\jbard\Desktop\Library\@CURSOS ACTUALES\@Data Analitycs\Python projects\Mineria-simulacion v2\data\processed/fact_paradas.csv"
ruta_log    = r"C:\Users\jbard\Desktop\Library\@CURSOS ACTUALES\@Data Analitycs\Python projects\Mineria-simulacion v2\data\processed/_log_exportaciones.csv"

fact_paradas_final.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

registro = pd.DataFrame([{
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "archivo": "fact_paradas.csv",
    "filas": len(fact_paradas_final),
    "columnas": len(fact_paradas_final.columns),
    "filas_por_planta": str(conteo_real_paradas),
    "duplicados_por_planta": str(duplicados_por_planta_paradas),
    "conflictos_por_planta": str(conflictos_por_planta_paradas),
    "dup_cruzados": int(dup_cruzados_paradas),
    "nulos_totales": int(fact_paradas_final.isna().sum().sum()),
    "hash_contenido": int(pd.util.hash_pandas_object(fact_paradas_final).sum()),
}])

registro.to_csv(ruta_log, mode="a", index=False,
                 header=not os.path.exists(ruta_log), encoding="utf-8-sig")

print(f"✅ Exportado: {ruta_salida}")
print(f"📋 Log actualizado: {ruta_log}")

✅ Exportado: C:\Users\jbard\Desktop\Library\@CURSOS ACTUALES\@Data Analitycs\Python projects\Mineria-simulacion v2\data\processed/fact_paradas.csv
📋 Log actualizado: C:\Users\jbard\Desktop\Library\@CURSOS ACTUALES\@Data Analitycs\Python projects\Mineria-simulacion v2\data\processed/_log_exportaciones.csv


## 3) FACT_PRODUCCION_LEGACY

In [140]:
directory =  r'C:\Users\jbard\Desktop\Library\@CURSOS ACTUALES\@Data Analitycs\Python projects\Mineria-simulacion v2'
prodLegacy_PC= pd.read_csv(directory+r'\data\raw\fact_produccion_PC_legacy.csv')
prodLegacy_PS= pd.read_csv(directory+r'\data\raw\fact_produccion_PS_legacy.csv')
prodLegacy_PN= pd.read_csv(directory+r'\data\raw\fact_produccion_PN_legacy.csv')

dim_plantas= pd.read_csv(directory+r'\data\data_master\dim_plantas.csv')
dim_equipos= pd.read_csv(directory+r'\data\data_master\dim_equipos.csv')
display(dim_equipos.head(10))
display(dim_plantas.head(3))

#Dataframes con datos muy sucios, se procede a inspeccionar de forma manual cada uno, proceder de forma masiva no parece ser eficiente

,equipo_id,planta_id,nombre_equipo,area,tipo_equipo,capacidad_nominal_tmh,anio_instalacion,clase_confiabilidad
0,PN-EQ-01,PN,Chancadora Primaria,Chancado,Chancadora,520.0,2013,media
1,PN-EQ-02,PN,Chancadora Secundaria,Chancado,Chancadora,430.0,2013,media
2,PN-EQ-03,PN,Chancadora Terciaria,Chancado,Chancadora,380.0,2016,alta
3,PN-EQ-04,PN,Molino SAG,Molienda,Molino,460.0,2011,baja
4,PN-EQ-05,PN,Molino de Bolas 1,Molienda,Molino,300.0,2011,baja
5,PN-EQ-06,PN,Molino de Bolas 2,Molienda,Molino,300.0,2015,media
6,PN-EQ-07,PN,Celda de Flotación,Flotación,Flotacion,350.0,2014,alta
7,PN-EQ-08,PN,Espesador de Relave,Espesado,Espesador,400.0,2012,media
8,PN-EQ-09,PN,Filtro Prensa,Filtrado,Filtro,260.0,2017,alta
9,PN-EQ-10,PN,Faja Transportadora 3,Chancado,Faja,500.0,2013,baja


,planta_id,nombre_planta,region,tipo_clima,factor_capacidad,anio_inicio_operacion
0,PN,Planta Norte,Costa,"Costa (seco, riesgo El Niño)",1.0,2015
1,PC,Planta Centro,Sierra Centro,Sierra (lluvias fuertes ene-mar),0.8,2012
2,PS,Planta Sur,Sierra Sur,Sierra alta (lluvias moderadas),1.2,2019


In [141]:
display(reporte_calidad(prodLegacy_PC,nombre_df='fact_mtto_PC'))
display(reporte_calidad(prodLegacy_PS,nombre_df='fact_mtto_PS'))
display(reporte_calidad(prodLegacy_PN,nombre_df='fact_mtto_PN'))


--- Resumen: fact_mtto_PC ---
Total filas del df: 4374
Sumatoria filas duplicadas: 0 (0.00%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,Horas_Parada,float64,0.0,4204,170,3.89,164
1,Ton_Rechazo,float64,0.0,4244,130,2.97,1927
2,Cod_Equipo,str,0.0,4374,0,0.00,70
3,Fecha,str,0.0,4374,0,0.00,427
4,Horas_Trab,float64,0.0,4374,0,0.00,169
5,Ton_Producidas,float64,0.0,4374,0,0.00,4095
6,Cap_Nominal_TMH,float64,0.0,4374,0,0.00,10
7,Fecha_Carga,str,0.0,4374,0,0.00,426



--- Resumen: fact_mtto_PS ---
Total filas del df: 2015
Sumatoria filas duplicadas: 0 (0.00%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,Horas_Parada,float64,0.0,1946,69,3.42,132
1,Ton_Rechazo,float64,0.0,1960,55,2.73,1440
2,Cod_Equipo,str,0.0,2015,0,0.00,70
3,Fecha,str,0.0,2015,0,0.00,202
4,Horas_Trab,float64,0.0,2015,0,0.00,135
5,Ton_Producidas,float64,0.0,2015,0,0.00,1941
6,Cap_Nominal_TMH,float64,0.0,2015,0,0.00,10
7,Fecha_Carga,str,0.0,2015,0,0.00,198



--- Resumen: fact_mtto_PN ---
Total filas del df: 7616
Sumatoria filas duplicadas: 0 (0.00%)
Columnas: 8


,columna,tipo_dato,duplicados,no_nulos,nulos,pct_nulos,unicos
0,Horas_Parada,float64,0.0,7327,289,3.79,188
1,Ton_Rechazo,float64,0.0,7385,231,3.03,2743
2,Cod_Equipo,str,0.0,7616,0,0.00,70
3,Fecha,str,0.0,7616,0,0.00,740
4,Horas_Trab,float64,0.0,7616,0,0.00,190
5,Ton_Producidas,float64,0.0,7616,0,0.00,7064
6,Cap_Nominal_TMH,float64,0.0,7616,0,0.00,10
7,Fecha_Carga,str,0.0,7616,0,0.00,740


#### Cod_equipo

In [142]:
vc_cod_equi = prodLegacy_PC['Cod_Equipo'].value_counts()  # solo calcula, no muestra

with pd.option_context('display.max_rows',8000):
    display(vc_cod_equi)   # Deberia mostrar completo, sin embargo no lo hace

print('La razon del trucanmiento es a :', vc_cod_equi.sum(),'cantidad de valores distintos')
#Podemos visualizar que el formato  de los equipos es un caos, se debe aplicar normalizacion con REGEX y definir el formato valido a usar
# PC - EQ - XX

Cod_Equipo
PC_EQ_03      83
PC-EQ-01\t    77
PC-EQ-06      77
PC-EQ-08      76
PC-EQ-04\t    74
PCEQ06        70
 PC-EQ-08     70
PC-EQ-01      70
PC_EQ_10      70
PCEQ04        70
pc-eq-07      70
PC-EQ-10      70
PC-EQ-07\t    69
PC-EQ-05\t    69
 PC-EQ-04     69
PC-EQ-02      69
 PC-EQ-09     68
PC-EQ-02      68
PC-EQ-09      68
PC-EQ-10      68
PC-EQ-02\t    67
pc-eq-09      67
PC-EQ-10\t    67
PC-EQ-09\t    65
PC-EQ-05      65
pc-eq-08      65
pc-eq-05      64
PC-EQ-06\t    63
PC-EQ-04      63
PCEQ01        63
PC-EQ-07      63
PCEQ07        62
PC-EQ-03\t    62
PC-EQ-03      62
PC-EQ-07      61
PC_EQ_05      60
pc-eq-01      60
PC-EQ-03      60
PCEQ08        60
 PC-EQ-02     60
 PC-EQ-05     60
pc-eq-06      59
PCEQ09        59
PC-EQ-01      59
PCEQ10        59
PCEQ03        59
PC_EQ_06      58
PC-EQ-05      58
PC-EQ-06      58
PC_EQ_02      58
PCEQ05        58
 PC-EQ-01     58
pc-eq-03      58
 PC-EQ-07     58
PC-EQ-04      58
pc-eq-02      57
PC-EQ-08      57
PC_EQ_09      56
 PC

La razon del trucanmiento es a : 4374 cantidad de valores distintos


In [143]:

def normalizar_equipo_id(serie):
    return(serie.str.strip()
                .str.upper()
                .str.replace('_','-',regex=False)
                .str.replace(r"^([A-Z]{2})(EQ)" , r'\1-\2' ,regex=True )  #salida es PC-EQXX --Inicia con un numero de la A a la Z dos veces (valor1) EQ(valor2) , pon un - entre ellos
                .str.replace( r'EQ(\d)' , r'EQ-\1',regex=True ) # salida es PC-EQ-01  --EQ pegado EQ(/d) toca separarlos con - dentro de replace con -
           )


prodLegacy_PC['Cod_Equipo'] = normalizar_equipo_id(prodLegacy_PC['Cod_Equipo'])


display(prodLegacy_PC['Cod_Equipo'].value_counts())
print('Cantidad de equipos:',prodLegacy_PC['Cod_Equipo'].nunique())

# Columna Cod_Equipo normalizado y sin error tal como en la tabla de dimensiones de equipos

Cod_Equipo
PC-EQ-06    441
PC-EQ-01    440
PC-EQ-04    439
PC-EQ-03    439
PC-EQ-10    438
PC-EQ-07    438
PC-EQ-09    437
PC-EQ-08    436
PC-EQ-05    434
PC-EQ-02    432
Name: count, dtype: int64

Cantidad de equipos: 10


#### Horas_trabj

In [194]:
# Verificar si las horas de parada por encima del promedio son una gran cantidad o son pocos
mas_above_avg = prodLegacy_PC['Horas_Parada'].mean()
df_horas_above_avg = prodLegacy_PC[prodLegacy_PC['Horas_Parada'] > mas_above_avg]
pct_above_avg = round(len(horas_above_avg) / len(prodLegacy_PC) * 100, 2)


print(f"Promedio Horas_Parada: {promedio_horas_parada:.2f}", 'Horas')
print(f"Filas por encima del promedio: {len(produccion_above_avg)} de {len(prodLegacy_PC)} ({pct_above_avg}%) ; Shape df_horas_above_avg:{df_horas_above_avg.shape}") # len= n* registros
display(produccion_above_avg.head(5))

# Promedio de 'Horas_Parada' total
h_above_avg_team = produccion_above_avg.groupby('Cod_Equipo')['Horas_Parada'].mean().reset_index()
horas_above_avg_team= round(h_above_avg_team['Horas_Parada'].mean(),2)
print(f'La cantidad promedio de Horas_parada por equipo es de  "{horas_above_avg_team} Horas"  verificar si hay algun equipo que trabaja demasiadas horas y porqué')
h_above_avg_team

# Promedio de 'Horas_parada' por equipo


Promedio Horas_Parada: 1.44 Horas
Filas por encima del promedio: 775 de 4374 (17.72%) ; Shape df_horas_above_avg:(775, 8)


,Fecha,Cod_Equipo,Horas_Trab,Horas_Parada,Ton_Producidas,Ton_Rechazo,Cap_Nominal_TMH,Fecha_Carga
1,26/09/2024,PC-EQ-06,22.4,1.6,4701.6,39.1,240.0,2024-09-27 00:00:00
24,16/04/2024,PC-EQ-08,22.3,1.7,5906.4,47.8,320.0,2024-04-17 00:00:00
52,24/05/2024,PC-EQ-10,21.9,2.1,7945.2,90.9,400.0,2024-05-25 00:00:00
54,01/10/2024,PC-EQ-09,13.7,10.3,2656.8,17.0,208.0,2024-10-02 00:00:00
60,04/04/2024,PC-EQ-08,4.0,20.0,1139.3,24.0,320.0,2024-04-05 00:00:00


La cantidad promedio de Horas_parada por equipo es de  "7.4 Horas"  verificar si hay algun equipo que trabaja demasiadas horas y porqué


,Cod_Equipo,Horas_Parada
0,PC-EQ-01,7.288889
1,PC-EQ-02,8.194048
2,PC-EQ-03,5.974194
3,PC-EQ-04,8.963717
4,PC-EQ-05,9.615464
5,PC-EQ-06,8.469118
6,PC-EQ-07,6.294118
7,PC-EQ-08,7.592593
8,PC-EQ-09,5.047273
9,PC-EQ-10,6.580723
